# ML-07 — Refresh Opportunity Baseline and Top-10 Review

**Lane lock: Refresh / Content Opportunity Scoring.** This is the lane selected in ML-03 and used for the March data contract. This notebook builds a human-review baseline, not an automatic editing decision.

The decision moment is **March 15, 2026**. First-half data (March 1–15) can inform the rule. March 16–31 is used only to check whether earlier signals relate to subsequent observed demand; it is never an input to the score or ranked queue. June 2026 remains sealed.

## 1. Check two signals, then state one rule

**Signal check 1 — volume (quick-win flag context):** bucket eligible items by first-half impressions, and show `n` plus median second-half impressions. Verdict: **CONFIRMED** only if median later impressions rise in every successive bucket; otherwise **MIXED**. This checks that high-visibility candidates carry more observed demand, not that a refresh will succeed.

**Signal check 2 — CTR versus position (CTR-fix flag context):** show low-CTR counts and shares by first-half average-position tier. Verdict: **CONFIRMED** if the low-CTR share differs by at least 10 percentage points across sufficiently populated tiers; otherwise **MIXED**. Position-aware thresholds matter because CTR varies with rank.

**One rule:** for items with at least 500 first-half impressions, average position from 1 through 20, and first-half CTR below 0.5%, rank by impression-weighted CTR gap: `impressions × (0.5 − CTR percent) / 100`. Every candidate gets exactly one reason code, `low_ctr_visible_page`, and one action label, `review_title_and_snippet`. The score is a prioritization heuristic, not a prediction of recoverable clicks.

In [2]:
import getpass
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and "google.colab" in sys.modules:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input is hidden): ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face READ token is required to query the gated warehouse.")

con = duckdb.connect()
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token, HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

monthly = con.sql(f"""
    WITH per_item AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_first_half,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_first_half,
               AVG(CASE WHEN report_date < DATE '2026-03-16'
                         AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
               SUM(CASE WHEN report_date >= DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_second_half
        FROM {FACT_MARCH}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *,
           100.0 * clicks_first_half / NULLIF(impressions_first_half, 0) AS ctr_first_half_pct
    FROM per_item
    WHERE impressions_first_half >= 100
""").df()

if monthly.empty:
    raise ValueError("The March slice contains no items with at least 100 first-half impressions.")

monthly["volume_bucket"] = pd.cut(
    monthly["impressions_first_half"],
    bins=[100, 500, 2000, float("inf")],
    labels=["100-499", "500-1,999", "2,000+"],
    right=False,
)
volume_table = (
    monthly.groupby("volume_bucket", observed=False)
    .agg(n=("impressions_first_half", "size"),
         median_first_half_impressions=("impressions_first_half", "median"),
         median_second_half_impressions=("impressions_second_half", "median"))
    .reset_index()
)
volume_outcomes = volume_table.loc[volume_table["n"] > 0, "median_second_half_impressions"].tolist()
volume_verdict = "CONFIRMED" if len(volume_outcomes) >= 2 and all(a < b for a, b in zip(volume_outcomes, volume_outcomes[1:])) else "MIXED"
print("Signal check 1 — volume / quick-win flag context. One row per eligible item; n is shown:")
display(volume_table)
print(f"Volume verdict: {volume_verdict} — later median impressions {'rise with' if volume_verdict == 'CONFIRMED' else 'do not rise monotonically with'} first-half volume. This is demand context, not evidence that refreshing works.")

monthly["position_tier"] = pd.cut(
    monthly["avg_position_first_half"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    right=True,
).astype("string").fillna("position_unavailable")
monthly["low_ctr"] = monthly["ctr_first_half_pct"] < 0.5
position_table = (
    monthly.groupby("position_tier", observed=False)
    .agg(n=("low_ctr", "size"),
         low_ctr_n=("low_ctr", "sum"),
         low_ctr_share=("low_ctr", "mean"),
         median_ctr_pct=("ctr_first_half_pct", "median"))
    .reset_index()
)
populated_position = position_table.loc[position_table["n"] >= 50, "low_ctr_share"]
position_verdict = "CONFIRMED" if len(populated_position) >= 2 and populated_position.max() - populated_position.min() >= 0.10 else "MIXED"
print("\nSignal check 2 — CTR versus position / CTR-fix flag context. CTR is percent, not a 0–1 fraction:")
display(position_table)
print(f"CTR-position verdict: {position_verdict} — {'low-CTR incidence varies materially across populated position tiers' if position_verdict == 'CONFIRMED' else 'the tier differences do not cross the pre-set 10-point threshold'}; position is retained as a guard in the rule.")

Signal check 1 — volume / quick-win flag context. One row per eligible item; n is shown:


,volume_bucket,n,median_first_half_impressions,median_second_half_impressions
0,100-499,35724,222.0,243.0
1,"500-1,999",26456,919.0,1023.0
2,"2,000+",15360,3789.0,4064.0


Volume verdict: CONFIRMED — later median impressions rise with first-half volume. This is demand context, not evidence that refreshing works.

Signal check 2 — CTR versus position / CTR-fix flag context. CTR is percent, not a 0–1 fraction:


,position_tier,n,low_ctr_n,low_ctr_share,median_ctr_pct
0,1-3,9197,6455,0.701859,0.265554
1,11-20,15306,12546,0.819679,0.097967
2,21+,15590,14299,0.917191,0.000000
3,4-10,37447,28930,0.772559,0.182288


CTR-position verdict: CONFIRMED — low-CTR incidence varies materially across populated position tiers; position is retained as a guard in the rule.


## 2. Build and write the ranked review queue

The queue uses first-half measurements only. Items that do not meet the single rule are not queue candidates; every saved row gets the same one reason code and action. The generated CSV is intentionally git-ignored as a dataset artifact; rerun this notebook to regenerate it.

In [3]:
candidates = monthly.loc[
    (monthly["impressions_first_half"] >= 500)
    & monthly["avg_position_first_half"].between(1, 20, inclusive="both")
    & (monthly["ctr_first_half_pct"] < 0.5)
].copy()

if candidates.empty:
    raise ValueError("The predeclared low-CTR visible-page rule produced no candidates in March.")

candidates["baseline_score"] = (
    candidates["impressions_first_half"]
    * (0.5 - candidates["ctr_first_half_pct"])
    / 100.0
)
candidates["reason_code"] = "low_ctr_visible_page"
candidates["action"] = "review_title_and_snippet"
candidates = candidates.sort_values(
    ["baseline_score", "impressions_first_half"], ascending=[False, False]
).reset_index(drop=True)
candidates.insert(0, "baseline_rank", range(1, len(candidates) + 1))

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "work" / "notebooks" / "w04_baseline_score.ipynb").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate this repository's work/notebooks/w04_baseline_score.ipynb.")
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue_columns = [
    "baseline_rank", "client_hash_id", "content_hash_id", "baseline_score",
    "reason_code", "action", "impressions_first_half", "clicks_first_half",
    "ctr_first_half_pct", "avg_position_first_half",
]
candidates[queue_columns].to_csv(output_path, index=False)

print(f"Eligible items checked: {len(monthly):,}")
print(f"Queue candidates: {len(candidates):,}")
print(f"Rule: {candidates['reason_code'].iloc[0]} → {candidates['action'].iloc[0]}")
print(f"CSV written to: {output_path.relative_to(repo_root)}")
print("The CSV contains pseudonymous IDs for authorized item lookup; IDs are not printed in the notebook preview.")

Eligible items checked: 77,540
Queue candidates: 26,276
Rule: low_ctr_visible_page → review_title_and_snippet
CSV written to: work\outputs\baseline_action_score.csv
The CSV contains pseudonymous IDs for authorized item lookup; IDs are not printed in the notebook preview.


## 3. Review the top ten skeptically

Each line states the action, why the rule selected it, and what could make the recommendation wrong. Row numbers are queue ranks only; pseudonymous IDs are not displayed.

In [4]:
def review_caveat(row):
    if row["impressions_first_half"] < 700:
        return "Low-end eligible volume makes the CTR estimate noisy; more impressions could move it above the cutoff."
    if row["ctr_first_half_pct"] >= 0.45:
        return "CTR is close to the 0.5% threshold; sampling variation could remove the flag."
    if row["avg_position_first_half"] > 10:
        return "Average position is near the lower end of the rule; rank variation may explain the low CTR."
    return "SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem."

top_ten = candidates.head(10).copy()
if len(top_ten) < 10:
    raise ValueError(f"Only {len(top_ten)} candidates matched the predeclared rule; cannot complete a top-ten review.")

for row in top_ten.itertuples(index=False):
    record = row._asdict()
    wrong_if = review_caveat(record)
    print(
        f"Rank {record['baseline_rank']}: {record['action']} — "
        f"{record['reason_code']}; {record['impressions_first_half']:,.0f} first-half impressions, "
        f"CTR {record['ctr_first_half_pct']:.3f}%, position {record['avg_position_first_half']:.1f}. "
        f"Could be wrong if: {wrong_if}"
    )

Rank 1: review_title_and_snippet — low_ctr_visible_page; 83,772 first-half impressions, CTR 0.000%, position 8.6. Could be wrong if: SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem.
Rank 2: review_title_and_snippet — low_ctr_visible_page; 86,860 first-half impressions, CTR 0.059%, position 5.8. Could be wrong if: SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem.
Rank 3: review_title_and_snippet — low_ctr_visible_page; 143,173 first-half impressions, CTR 0.247%, position 16.0. Could be wrong if: Average position is near the lower end of the rule; rank variation may explain the low CTR.
Rank 4: review_title_and_snippet — low_ctr_visible_page; 73,639 first-half impressions, CTR 0.024%, position 2.8. Could be wrong if: SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem.
Rank 5: review_title

## 4. Weak picks and leakage check

The weakest top-ten picks are those with the smallest score cushion above the hard CTR or minimum-volume cutoff. Their ranking is fragile: a small change in the observed CTR or position measurement can remove them. A CTR below 0.5% does not prove a title or snippet is the cause; search intent, SERP features, device mix, and sampling noise are plausible alternatives.

Leakage check: the score uses only March 1–15 impressions, clicks, CTR, and position. March 16–31 impressions appear only in signal-check tables, not the candidates' score or CSV. No decline proxy, label, product flag, client ID, or future-window field enters the rule. The output remains decision-support for a human reviewer.

In [5]:
weakest_top_ten = top_ten.nsmallest(3, "baseline_score")
print("Three weakest top-ten picks by score, with IDs omitted:")
for row in weakest_top_ten.itertuples(index=False):
    print(
        f"Rank {row.baseline_rank}: score {row.baseline_score:.4f}; "
        f"CTR {row.ctr_first_half_pct:.3f}% versus 0.5% cutoff; "
        f"{row.impressions_first_half:,.0f} impressions. "
        f"Weakness: {review_caveat(row._asdict())}"
    )

score_inputs = {"impressions_first_half", "ctr_first_half_pct", "avg_position_first_half"}
forbidden_inputs = {
    "impressions_second_half", "is_declining_proxy", "trend_direction", "trend_pct",
    "action_type", "health_score", "priority_score", "ga4_sessions",
}
assert score_inputs.isdisjoint(forbidden_inputs)
assert set(candidates["reason_code"]) == {"low_ctr_visible_page"}
assert set(candidates["action"]) == {"review_title_and_snippet"}
assert candidates["baseline_score"].is_monotonic_decreasing
print("Leakage/action checks passed: one reason code, one action, and only pre-decision signals in the score.")

Three weakest top-ten picks by score, with IDs omitted:
Rank 10: score 256.7000; CTR 0.148% versus 0.5% cutoff; 72,940 impressions. Weakness: Average position is near the lower end of the rule; rank variation may explain the low CTR.
Rank 9: score 263.4000; CTR 0.027% versus 0.5% cutoff; 55,680 impressions. Weakness: SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem.
Rank 8: score 272.3700; CTR 0.202% versus 0.5% cutoff; 91,474 impressions. Weakness: SERP layout, query intent, device mix, or seasonality may explain CTR; the rule does not establish a snippet problem.
Leakage/action checks passed: one reason code, one action, and only pre-decision signals in the score.


## 5. Self-check

- [x] Refresh / Content Opportunity Scoring is the selected lane.
- [x] Two signal checks include bucket tables with `n`, one linked to volume/quick-win and one to CTR/position.
- [x] One rule emits a score, exactly one reason code, and an action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv` (git-ignored by design).
- [x] The top ten each have an action, rationale, and a specific way the pick could be wrong.
- [x] No future-window or label-derived fields enter the rule.
- [ ] Run top to bottom with approved warehouse access; inspect tables and all ten reviews.
- [x] No raw client names, URLs, or private queries are displayed.
- [ ] Commit this notebook to your public repo, then submit the repository URL on the card.